In [4]:
import pandas as pd
import urllib.request
import zipfile
import io
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Download Dataset from UCI (official source)
print("Downloading dataset...")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
response = urllib.request.urlopen(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.read()))
zip_file.extractall("/tmp/spam/")
print("Downloaded successfully!\n")

# 2. Load the file
df = pd.read_csv(
    "/tmp/spam/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="latin-1"
)

print("Dataset shape :", df.shape)
print("Labels        :", df["label"].unique())
print(df["label"].value_counts())
print()

# 3. Encode Labels (spam=1, ham=0)
df["label"] = (df["label"].str.strip() == "spam").astype(int)

# 4. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"],
    test_size=0.2, random_state=42, stratify=df["label"]
)

# 5. TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

# 6. Train Naive Bayes
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_vec, y_train)
nb_pred = nb.predict(X_test_vec)

# 7. Train Logistic Regression
lr = LogisticRegression(max_iter=1000, C=5)
lr.fit(X_train_vec, y_train)
lr_pred = lr.predict(X_test_vec)

# 8. Show Accuracy
nb_acc = round(accuracy_score(y_test, nb_pred) * 100, 2)
lr_acc = round(accuracy_score(y_test, lr_pred) * 100, 2)

print("=" * 45)
print("       MODEL ACCURACY RESULTS")
print("=" * 45)
print("  Naive Bayes         :", nb_acc, "%")
print("  Logistic Regression :", lr_acc, "%")
print("=" * 45)
print()
print("-- Naive Bayes Report --")
print(classification_report(y_test, nb_pred, target_names=["Ham", "Spam"]))
print("-- Logistic Regression Report --")
print(classification_report(y_test, lr_pred, target_names=["Ham", "Spam"]))

# 9. User Input Loop
print("=" * 45)
print("  SPAM DETECTOR - Enter a message")
print("  (type 'exit' to quit)")
print("=" * 45)

while True:
    msg = input("\nEnter message: ").strip()

    if msg.lower() == "exit":
        print("Goodbye!")
        break

    if msg == "":
        print("Please enter a message.")
        continue

    vec = tfidf.transform([msg])
    nb_res = "SPAM" if nb.predict(vec)[0] == 1 else "HAM"
    lr_res = "SPAM" if lr.predict(vec)[0] == 1 else "HAM"
    nb_prob = round(nb.predict_proba(vec)[0][1] * 100, 1)
    lr_prob = round(lr.predict_proba(vec)[0][1] * 100, 1)

    print("  Naive Bayes         ->", nb_res, "| spam chance:", nb_prob, "%")
    print("  Logistic Regression ->", lr_res, "| spam chance:", lr_prob, "%")

Downloaded successfully!

Dataset shape : (5572, 2)
Labels        : ['ham' 'spam']
label
ham     4825
spam     747
Name: count, dtype: int64

       MODEL ACCURACY RESULTS
  Naive Bayes         : 98.57 %
  Logistic Regression : 98.03 %

-- Naive Bayes Report --
              precision    recall  f1-score   support

         Ham       0.99      1.00      0.99       966
        Spam       0.99      0.91      0.94       149

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115

-- Logistic Regression Report --
              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       966
        Spam       0.99      0.86      0.92       149

    accuracy                           0.98      1115
   macro avg       0.99      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115

  SPAM DETECTOR - Enter a message
  (type 'exit' to

In [6]:
from google.colab import files
uploaded = files.upload()  # select SMSSpamCollection file

df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="latin-1"
)

print(df.head(10))
print(df["label"].value_counts())

Saving SMSSpamCollection to SMSSpamCollection
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
5  spam  FreeMsg Hey there darling it's been 3 week's n...
6   ham  Even my brother is not like to speak with me. ...
7   ham  As per your request 'Melle Melle (Oru Minnamin...
8  spam  WINNER!! As a valued network customer you have...
9  spam  Had your mobile 11 months or more? U R entitle...
label
ham     4825
spam     747
Name: count, dtype: int64
